# Capa 1: Creacion de Base de Datos - EcoMarket

Este notebook crea la base de datos SQLite con datos ficticios para el chatbot.

**Tablas:**
- productos - Catalogo con stock, precio y categorias
- pedidos - Historial de pedidos por cliente (email)
- promociones - Ofertas vigentes vinculadas a productos
- devoluciones - Solicitudes de devolucion
- tickets_soporte - Casos escalados a agente humano

**Salida:** ../data/ecomarket.db

**Recarga:** por defecto `RESET_DB = False` conserva pedidos reales y solo actualiza catálogo/promos. Pon `RESET_DB = True` si quieres borrar todo.

In [1]:
import sqlite3
import os
from datetime import datetime, timedelta
import random
import pandas as pd

In [2]:
# Crear directorio data si no existe
os.makedirs("../data", exist_ok=True)

# False = conserva pedidos, devoluciones y tickets al recargar (solo actualiza catálogo y promos)
# True  = borra todo y regenera la BD desde cero
RESET_DB = False

DB_PATH = "../data/ecomarket.db"
if RESET_DB and os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("BD anterior eliminada (RESET_DB=True)")

# Conectar (crea el archivo si no existe)
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
modo = "regeneración completa" if RESET_DB else "actualización (pedidos conservados)"
print(f"Base de datos en: {DB_PATH} [{modo}]")

Base de datos en: ../data/ecomarket.db [actualización (pedidos conservados)]


## 1. Creacion de tablas

In [3]:
# Tabla PRODUCTOS
cursor.execute("""CREATE TABLE IF NOT EXISTS productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    categoria TEXT NOT NULL,
    precio REAL NOT NULL,
    stock INTEGER NOT NULL,
    descripcion TEXT,
    unidad TEXT DEFAULT "unidad"
)""")

# Tabla PEDIDOS
cursor.execute("""CREATE TABLE IF NOT EXISTS pedidos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    email_cliente TEXT NOT NULL,
    fecha_pedido TEXT NOT NULL,
    estado TEXT NOT NULL,
    total REAL NOT NULL,
    direccion_envio TEXT,
    fecha_entrega_estimada TEXT
)""")

# Tabla DETALLE_PEDIDO
cursor.execute("""CREATE TABLE IF NOT EXISTS detalle_pedido (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    pedido_id INTEGER NOT NULL,
    producto_id INTEGER NOT NULL,
    cantidad INTEGER NOT NULL,
    precio_unitario REAL NOT NULL,
    FOREIGN KEY (pedido_id) REFERENCES pedidos(id),
    FOREIGN KEY (producto_id) REFERENCES productos(id)
)""")

# Tabla PROMOCIONES
cursor.execute("""CREATE TABLE IF NOT EXISTS promociones (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    producto_id INTEGER NOT NULL,
    descripcion TEXT NOT NULL,
    descuento_porcentaje REAL NOT NULL,
    fecha_inicio TEXT NOT NULL,
    fecha_fin TEXT NOT NULL,
    activa INTEGER DEFAULT 1,
    FOREIGN KEY (producto_id) REFERENCES productos(id)
)""")

# Tabla DEVOLUCIONES
cursor.execute("""CREATE TABLE IF NOT EXISTS devoluciones (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    pedido_id INTEGER NOT NULL,
    email_cliente TEXT NOT NULL,
    motivo TEXT NOT NULL,
    estado TEXT DEFAULT "pendiente",
    fecha_solicitud TEXT NOT NULL,
    fecha_resolucion TEXT,
    comentarios TEXT,
    FOREIGN KEY (pedido_id) REFERENCES pedidos(id)
)""")

# Tabla TICKETS_SOPORTE
cursor.execute("""CREATE TABLE IF NOT EXISTS tickets_soporte (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    email_cliente TEXT NOT NULL,
    asunto TEXT NOT NULL,
    descripcion TEXT NOT NULL,
    estado TEXT DEFAULT "abierto",
    fecha_creacion TEXT NOT NULL,
    pedido_id INTEGER,
    FOREIGN KEY (pedido_id) REFERENCES pedidos(id)
)""")

conn.commit()
print("Tablas listas")

Tablas listas


## 2. Insercion de datos ficticios
### 2.1 Productos

In [4]:
productos = [
    # Frutas
    ("Manzana Roja", "Frutas", 2.50, 150, "Manzana roja importada, 1kg", "kg"),
    ("Platano", "Frutas", 1.80, 200, "Platano maduro, 1kg", "kg"),
    ("Aguacate", "Frutas", 4.50, 60, "Aguacate Hass maduro", "unidad"),
    ("Naranja", "Frutas", 2.00, 180, "Naranja de Valencia, 1kg", "kg"),
    ("Mango", "Frutas", 3.50, 80, "Mango fresco tropical", "unidad"),
    ("Pera", "Frutas", 2.20, 90, "Pera conferencia, 1kg", "kg"),
    ("Mandarina", "Frutas", 2.00, 100, "Mandarina clementina, 1kg", "kg"),
    # Verduras
    ("Tomate", "Verduras", 3.00, 100, "Tomate fresco de huerta, 1kg", "kg"),
    ("Lechuga", "Verduras", 1.50, 80, "Lechuga iceberg fresca", "unidad"),
    ("Zanahoria", "Verduras", 1.20, 120, "Zanahoria fresca, 1kg", "kg"),
    ("Papa", "Verduras", 2.00, 150, "Papa blanca fresca, 1kg", "kg"),
    ("Cebolla", "Verduras", 1.50, 130, "Cebolla blanca fresca, 1kg", "kg"),
    ("Pimentón", "Verduras", 2.80, 90, "Pimentón rojo fresco", "unidad"),
    ("Pepino", "Verduras", 1.80, 100, "Pepino fresco de huerta", "unidad"),
    # Lacteos
    ("Leche Entera", "Lacteos", 1.20, 300, "Leche entera 1L", "litro"),
    ("Yogur Natural", "Lacteos", 0.80, 200, "Yogur natural 125g", "unidad"),
    ("Queso Manchego", "Lacteos", 8.50, 40, "Queso manchego curado 250g", "unidad"),
    ("Mantequilla", "Lacteos", 2.30, 90, "Mantequilla sin sal 250g", "unidad"),
    ("Leche de Almendra", "Lacteos", 2.80, 70, "Leche de almendra 1L sin azucar", "litro"),
    ("Huevos", "Lacteos", 3.20, 120, "Huevos frescos de gallina, docena", "docena"),
    # Carnes
    ("Pechuga de Pollo", "Carnes", 6.90, 50, "Pechuga de pollo fresca, 1kg", "kg"),
    ("Carne Molida", "Carnes", 8.50, 40, "Carne molida de res, 500g", "unidad"),
    ("Salmon Fresco", "Carnes", 12.00, 30, "Filete de salmon fresco, 300g", "unidad"),
    ("Jamon Serrano", "Carnes", 15.00, 25, "Jamon serrano loncheado 200g", "unidad"),
    ("Chorizo", "Carnes", 4.50, 60, "Chorizo espanol 300g", "unidad"),
    ("Carne de Res", "Carnes", 12.00, 40, "Carne de res fresca para asar, 1kg", "kg"),
    # Panaderia
    ("Pan Integral", "Panaderia", 2.10, 100, "Pan integral rebanado 500g", "unidad"),
    ("Croissant", "Panaderia", 1.50, 80, "Croissant de mantequilla", "unidad"),
    ("Baguette", "Panaderia", 1.80, 60, "Baguette artesanal", "unidad"),
    ("Tortillas de Maiz", "Panaderia", 1.60, 90, "Tortillas de maiz x12", "paquete"),
    ("Pan de Pita", "Panaderia", 2.80, 75, "Pan de pita integral, paquete de 6 unidades", "paquete"),
    # Bebidas
    ("Agua Mineral", "Bebidas", 0.70, 500, "Agua mineral 1.5L", "botella"),
    ("Zumo de Naranja", "Bebidas", 2.50, 100, "Zumo de naranja natural 1L", "litro"),
    ("Coca-Cola", "Bebidas", 1.80, 200, "Coca-Cola 2L", "botella"),
    ("Cerveza Artesanal", "Bebidas", 3.50, 80, "Cerveza IPA artesanal 330ml", "botella"),
    ("Cafe Molido", "Bebidas", 5.00, 60, "Cafe molido Colombia 250g", "unidad"),
    ("Te Verde", "Bebidas", 3.20, 70, "Te verde organico x20 sobres", "caja"),
    # Limpieza y Hogar
    ("Detergente Liquido", "Limpieza y Hogar", 4.50, 80, "Detergente liquido 2L", "botella"),
    ("Papel Higienico", "Limpieza y Hogar", 6.00, 100, "Papel higienico x12 rollos", "paquete"),
    ("Jabon de Manos", "Limpieza y Hogar", 2.20, 120, "Jabon liquido de manos 500ml", "botella"),
    ("Lavavajillas", "Limpieza y Hogar", 3.00, 90, "Lavavajillas concentrado 750ml", "botella"),
    ("Suavizante", "Limpieza y Hogar", 3.80, 70, "Suavizante para ropa 2L", "botella"),
    # Despensa
    ("Arroz Basmati", "Despensa", 2.80, 110, "Arroz basmati 1kg", "kg"),
    ("Pasta Penne", "Despensa", 1.50, 130, "Pasta penne rigate 500g", "unidad"),
    ("Aceite de Oliva", "Despensa", 7.00, 50, "Aceite de oliva virgen extra 750ml", "botella"),
    ("Atun en Lata", "Despensa", 2.00, 150, "Atun en aceite de oliva 3x80g", "pack"),
    ("Chocolate Negro", "Despensa", 3.50, 60, "Chocolate negro 70% cacao 100g", "unidad"),
    ("Galletas Integrales", "Despensa", 2.80, 80, "Galletas integrales con avena 200g", "paquete"),
    ("Cereales", "Despensa", 4.20, 70, "Cereales integrales con frutos rojos 375g", "caja"),
    ("Miel", "Despensa", 5.50, 40, "Miel pura de flores 500g", "unidad"),
    # Congelados
    ("Pizza Margarita", "Congelados", 4.20, 85, "Pizza margarita congelada 400g", "unidad"),
    ("Helado Vainilla", "Congelados", 3.80, 70, "Helado de vainilla 1L", "litro"),
    ("Verduras Salteadas", "Congelados", 2.90, 95, "Mix de verduras salteadas 450g", "paquete"),
    ("Pescado Empanado", "Congelados", 5.50, 55, "Filetes de merluza empanados x4", "paquete"),
    ("Lasaña Bolognesa", "Congelados", 4.80, 60, "Lasaña bolognesa congelada 400g", "unidad"),
    ("Pizza Congelada", "Congelados", 5.50, 50, "Pizza congelada de queso y pepperoni", "unidad"),
    ("Papas Congeladas", "Congelados", 4.00, 60, "Papas prefritas congeladas, bolsa 1kg", "kg"),
    # Snacks
    ("Patatas Fritas", "Snacks", 1.90, 140, "Patatas fritas onduladas 150g", "paquete"),
    ("Frutos Secos Mix", "Snacks", 4.60, 75, "Mix de almendras, nueces y pasas 200g", "paquete"),
    ("Palomitas Microondas", "Snacks", 1.40, 110, "Palomitas mantequilla x3", "pack"),
    ("Barritas Energeticas", "Snacks", 3.20, 90, "Barritas de avena y chocolate x6", "caja"),
    # Cuidado Personal
    ("Champu Suave", "Cuidado Personal", 3.50, 100, "Champu suave todo tipo de cabello 400ml", "botella"),
    ("Pasta Dental", "Cuidado Personal", 2.10, 130, "Pasta dental blanqueadora 75ml", "unidad"),
    ("Desodorante Roll-on", "Cuidado Personal", 2.80, 95, "Desodorante roll-on 50ml", "unidad"),
    ("Crema Hidratante", "Cuidado Personal", 6.50, 65, "Crema hidratante corporal 250ml", "unidad"),
    ("Gel de Bano", "Cuidado Personal", 2.40, 85, "Gel de bano pH neutro 750ml", "botella"),
    # Mascotas
    ("Pienso Perro Adulto", "Mascotas", 12.50, 45, "Pienso seco para perro adulto 3kg", "saco"),
    ("Pienso Gato", "Mascotas", 9.80, 50, "Pienso seco para gato 2kg", "saco"),
    ("Snacks para Perro", "Mascotas", 3.90, 80, "Snacks dentales para perro 200g", "paquete"),
    # Infantil
    ("Panales Talla 3", "Infantil", 14.00, 40, "Panales talla 3 (4-9 kg) x44", "paquete"),
    ("Leche Infantil", "Infantil", 11.50, 35, "Leche de continuacion 800g", "lata"),
    ("Papillas de Frutas", "Infantil", 2.60, 70, "Papillas de frutas variadas x4", "pack"),
]

insertados = 0
actualizados = 0
for nombre, categoria, precio, stock, descripcion, unidad in productos:
    cursor.execute("SELECT id FROM productos WHERE nombre = ?", (nombre,))
    row = cursor.fetchone()
    if row:
        # No sobrescribimos stock: puede haber bajado por pedidos reales
        cursor.execute(
            "UPDATE productos SET categoria=?, precio=?, descripcion=?, unidad=? WHERE id=?",
            (categoria, precio, descripcion, unidad, row[0]),
        )
        actualizados += 1
    else:
        cursor.execute(
            "INSERT INTO productos (nombre, categoria, precio, stock, descripcion, unidad) VALUES (?, ?, ?, ?, ?, ?)",
            (nombre, categoria, precio, stock, descripcion, unidad),
        )
        insertados += 1

conn.commit()
print(f"Productos: {insertados} nuevos, {actualizados} actualizados ({insertados + actualizados} en catálogo)")

df_productos = pd.read_sql("SELECT * FROM productos", conn)
print(f"Categorias: {df_productos['categoria'].nunique()}")
print(df_productos['categoria'].value_counts())

Productos: 0 nuevos, 72 actualizados (72 en catálogo)
Categorias: 13
categoria
Despensa            8
Verduras            7
Frutas              7
Congelados          7
Lacteos             6
Bebidas             6
Carnes              6
Panaderia           5
Limpieza y Hogar    5
Cuidado Personal    5
Snacks              4
Mascotas            3
Infantil            3
Name: count, dtype: int64


### 2.2 Pedidos

In [5]:
num_pedidos = cursor.execute("SELECT COUNT(*) FROM pedidos").fetchone()[0]

if num_pedidos > 0 and not RESET_DB:
    print(f"Pedidos existentes conservados: {num_pedidos}")
else:
    clientes = [
        "maria.garcia@email.com",
        "carlos.lopez@email.com",
        "ana.martinez@email.com",
        "pedro.sanchez@email.com",
        "laura.fernandez@email.com",
        "juan.rodriguez@email.com",
        "sofia.ruiz@email.com",
        "diego.moreno@email.com",
        "elena.torres@email.com",
        "miguel.herrera@email.com",
        "carmen.vargas@email.com",
        "pablo.navarro@email.com",
        "isabel.romero@email.com",
        "antonio.jimenez@email.com",
    ]

    estados_pedido = ["confirmado", "en_preparacion", "enviado", "entregado", "cancelado"]
    direcciones = [
        "Calle Mayor 15, Madrid",
        "Av. Diagonal 200, Barcelona",
        "Gran Via 45, Valencia",
        "Calle Sierpes 8, Sevilla",
        "Paseo de Gracia 100, Barcelona",
        "Calle Alcala 78, Madrid",
        "Calle Colon 22, Alicante",
        "Plaza Espana 5, Malaga",
        "Calle Real 33, Zaragoza",
        "Av. de la Constitucion 12, Bilbao",
    ]

    random.seed(42)
    pedidos_data = []

    for i in range(50):
        email = random.choice(clientes)
        dias_atras = random.randint(0, 30)
        fecha = (datetime.now() - timedelta(days=dias_atras)).strftime("%Y-%m-%d")
        estado = random.choice(estados_pedido)
        total = round(random.uniform(15.0, 120.0), 2)
        direccion = random.choice(direcciones)
        fecha_entrega = (datetime.now() + timedelta(days=random.randint(1, 5))).strftime("%Y-%m-%d")
        pedidos_data.append((email, fecha, estado, total, direccion, fecha_entrega))

    cursor.executemany("""
        INSERT INTO pedidos (email_cliente, fecha_pedido, estado, total, direccion_envio, fecha_entrega_estimada)
        VALUES (?, ?, ?, ?, ?, ?)
    """, pedidos_data)

    conn.commit()
    print(f"{len(pedidos_data)} pedidos de ejemplo insertados")

df_pedidos = pd.read_sql("SELECT * FROM pedidos", conn)
print(df_pedidos["estado"].value_counts())

Pedidos existentes conservados: 52
estado
confirmado        14
enviado           13
en_preparacion    10
cancelado          8
entregado          7
Name: count, dtype: int64


### 2.3 Detalle de pedidos

In [6]:
num_detalles = cursor.execute("SELECT COUNT(*) FROM detalle_pedido").fetchone()[0]

if num_detalles > 0 and not RESET_DB:
    print(f"Detalle de pedidos conservado: {num_detalles} líneas")
else:
    random.seed(123)
    detalles = []
    num_productos = cursor.execute("SELECT COUNT(*) FROM productos").fetchone()[0]
    max_pedido_id = cursor.execute("SELECT MAX(id) FROM pedidos").fetchone()[0] or 0

    for pedido_id in range(1, min(51, max_pedido_id + 1)):
        n_items = random.randint(2, 7)
        productos_elegidos = random.sample(range(1, num_productos + 1), n_items)
        for prod_id in productos_elegidos:
            cantidad = random.randint(1, 4)
            cursor.execute("SELECT precio FROM productos WHERE id = ?", (prod_id,))
            precio = cursor.fetchone()[0]
            detalles.append((pedido_id, prod_id, cantidad, precio))

    cursor.executemany("""
        INSERT INTO detalle_pedido (pedido_id, producto_id, cantidad, precio_unitario)
        VALUES (?, ?, ?, ?)
    """, detalles)

    conn.commit()
    print(f"{len(detalles)} items de detalle insertados")

Detalle de pedidos conservado: 214 líneas


### 2.4 Promociones vigentes

In [7]:
hoy = datetime.now().strftime("%Y-%m-%d")
en_7_dias = (datetime.now() + timedelta(days=7)).strftime("%Y-%m-%d")
en_14_dias = (datetime.now() + timedelta(days=14)).strftime("%Y-%m-%d")
hace_3_dias = (datetime.now() - timedelta(days=3)).strftime("%Y-%m-%d")

def _id_producto(nombre: str) -> int:
    cursor.execute("SELECT id FROM productos WHERE nombre = ?", (nombre,))
    row = cursor.fetchone()
    if row is None:
        raise ValueError(f"Producto no encontrado: {nombre}")
    return row[0]

# Vincular por nombre (no por id fijo) para que no se desalineen al añadir productos
promociones_def = [
    ("Manzana Roja", "2x1 en Manzanas Rojas", 50.0, hace_3_dias, en_7_dias, 1),
    ("Leche Entera", "20% descuento en Leche Entera", 20.0, hoy, en_14_dias, 1),
    ("Pechuga de Pollo", "15% descuento en Pechuga de Pollo", 15.0, hoy, en_7_dias, 1),
    ("Agua Mineral", "Agua Mineral: 3x2", 33.0, hace_3_dias, en_14_dias, 1),
    ("Aceite de Oliva", "10% descuento en Aceite de Oliva", 10.0, hoy, en_7_dias, 1),
    ("Pan Integral", "Pan Integral a mitad de precio los martes", 50.0, hoy, en_14_dias, 1),
    ("Chocolate Negro", "25% descuento en Chocolate Negro", 25.0, hace_3_dias, en_7_dias, 1),
    ("Cerveza Artesanal", "Cerveza Artesanal: 4x3", 25.0, hoy, en_14_dias, 1),
    ("Pizza Margarita", "Pizza Margarita: 2da unidad al 50%", 50.0, hoy, en_7_dias, 1),
    ("Patatas Fritas", "Patatas Fritas: 3x2", 33.0, hace_3_dias, en_14_dias, 1),
    ("Champu Suave", "Champu Suave: 20% descuento", 20.0, hoy, en_14_dias, 1),
    ("Pienso Perro Adulto", "Pienso Perro: 10% descuento", 10.0, hoy, en_7_dias, 1),
    ("Panales Talla 3", "Panales: pack ahorro 15%", 15.0, hace_3_dias, en_14_dias, 1),
]

promociones = [
    (_id_producto(nombre), desc, pct, ini, fin, activa)
    for nombre, desc, pct, ini, fin, activa in promociones_def
]

cursor.execute("DELETE FROM promociones")
cursor.executemany("""
    INSERT INTO promociones (producto_id, descripcion, descuento_porcentaje, fecha_inicio, fecha_fin, activa)
    VALUES (?, ?, ?, ?, ?, ?)
""", promociones)

conn.commit()
print(f"{len(promociones)} promociones insertadas")

df_promos = pd.read_sql("""
    SELECT p.nombre, pr.descripcion, pr.descuento_porcentaje, pr.fecha_fin
    FROM promociones pr
    JOIN productos p ON p.id = pr.producto_id
    WHERE pr.activa = 1
""", conn)
df_promos

13 promociones insertadas


,nombre,descripcion,descuento_porcentaje,fecha_fin
0,Manzana Roja,2x1 en Manzanas Rojas,50.0,2026-07-01
1,Leche Entera,20% descuento en Leche Entera,20.0,2026-07-08
2,Pechuga de Pollo,15% descuento en Pechuga de Pollo,15.0,2026-07-01
3,Agua Mineral,Agua Mineral: 3x2,33.0,2026-07-08
4,Aceite de Oliva,10% descuento en Aceite de Oliva,10.0,2026-07-01
5,Pan Integral,Pan Integral a mitad de precio los martes,50.0,2026-07-08
6,Chocolate Negro,25% descuento en Chocolate Negro,25.0,2026-07-01
7,Cerveza Artesanal,Cerveza Artesanal: 4x3,25.0,2026-07-08
8,Pizza Margarita,Pizza Margarita: 2da unidad al 50%,50.0,2026-07-01
9,Patatas Fritas,Patatas Fritas: 3x2,33.0,2026-07-08


### 2.5 Devoluciones de ejemplo

In [8]:
hace_5_dias = (datetime.now() - timedelta(days=5)).strftime("%Y-%m-%d")
hace_2_dias = (datetime.now() - timedelta(days=2)).strftime("%Y-%m-%d")

num_devoluciones = cursor.execute("SELECT COUNT(*) FROM devoluciones").fetchone()[0]

if num_devoluciones > 0 and not RESET_DB:
    print(f"Devoluciones existentes conservadas: {num_devoluciones}")
else:
    devoluciones = [
        (3, "ana.martinez@email.com", "Producto llego danado", "aprobada", hace_5_dias, hace_2_dias, "Reembolso procesado"),
        (7, "carlos.lopez@email.com", "Producto equivocado", "pendiente", hace_2_dias, None, None),
        (12, "laura.fernandez@email.com", "No corresponde a lo solicitado", "en_revision", hace_3_dias, None, "En espera de fotos"),
        (18, "pedro.sanchez@email.com", "Producto caducado al recibir", "aprobada", hace_5_dias, hace_2_dias, "Reembolso parcial aplicado"),
        (25, "elena.torres@email.com", "Envase roto en la entrega", "pendiente", hace_2_dias, None, None),
        (31, "miguel.herrera@email.com", "Calidad inferior a la esperada", "en_revision", hace_3_dias, None, "Equipo de calidad revisando"),
        (38, "carmen.vargas@email.com", "Pedido incompleto", "rechazada", hace_5_dias, hace_2_dias, "Faltaba un item pero el cliente ya lo consumio"),
        (44, "pablo.navarro@email.com", "Producto no fresco", "aprobada", hace_5_dias, hace_2_dias, "Sustitucion enviada"),
    ]

    cursor.executemany("""
        INSERT INTO devoluciones (pedido_id, email_cliente, motivo, estado, fecha_solicitud, fecha_resolucion, comentarios)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, devoluciones)

    conn.commit()
    print(f"{len(devoluciones)} devoluciones insertadas")

Devoluciones existentes conservadas: 8


### 2.6 Tickets de soporte

In [9]:
hace_10_dias = (datetime.now() - timedelta(days=10)).strftime("%Y-%m-%d %H:%M:%S")
hace_7_dias = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d %H:%M:%S")
hace_4_dias = (datetime.now() - timedelta(days=4)).strftime("%Y-%m-%d %H:%M:%S")
hace_1_dia = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S")
hoy_ticket = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

num_tickets = cursor.execute("SELECT COUNT(*) FROM tickets_soporte").fetchone()[0]

if num_tickets > 0 and not RESET_DB:
    print(f"Tickets existentes conservados: {num_tickets}")
else:
    tickets = [
        ("maria.garcia@email.com", "Pedido no recibido", "Realice un pedido hace 5 dias y aun no ha llegado. El estado dice enviado.", "abierto", hace_4_dias, 5),
        ("carlos.lopez@email.com", "Cobro duplicado", "Me han cobrado dos veces el mismo pedido en mi tarjeta.", "en_proceso", hace_7_dias, 7),
        ("ana.martinez@email.com", "Consulta alergenos", "Necesito saber si el yogur natural contiene lactosa o trazas de frutos secos.", "cerrado", hace_10_dias, None),
        ("juan.rodriguez@email.com", "Cambio de direccion", "Quiero cambiar la direccion de entrega de mi pedido en curso.", "abierto", hace_1_dia, 22),
        ("sofia.ruiz@email.com", "Producto agotado", "Compre leche de almendra pero llego agotada en tienda online, quiero alternativa.", "en_proceso", hace_4_dias, 15),
        ("elena.torres@email.com", "Factura no recibida", "Necesito la factura del pedido 25 para mi empresa.", "abierto", hoy_ticket, 25),
        ("miguel.herrera@email.com", "Problema con promocion", "La promo 2x1 en manzanas no se aplico en mi pedido.", "abierto", hace_1_dia, 28),
        ("isabel.romero@email.com", "Queja por demora", "La entrega llego 2 dias tarde y algunos productos estaban fuera de temperatura.", "cerrado", hace_10_dias, 33),
    ]

    cursor.executemany("""
        INSERT INTO tickets_soporte (email_cliente, asunto, descripcion, estado, fecha_creacion, pedido_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, tickets)

    conn.commit()
    print(f"{len(tickets)} tickets de soporte insertados")

Tickets existentes conservados: 8


## 3. Verificacion final

In [10]:
tablas = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print('Resumen de la base de datos EcoMarket')
print('=' * 45)
for (tabla,) in tablas:
    count = cursor.execute(f'SELECT COUNT(*) FROM {tabla}').fetchone()[0]
    print(f'  {tabla:<20} -> {count} registros')
print()
print('Base de datos lista para usar')

Resumen de la base de datos EcoMarket
  productos            -> 72 registros
  sqlite_sequence      -> 6 registros
  pedidos              -> 52 registros
  detalle_pedido       -> 214 registros
  promociones          -> 13 registros
  devoluciones         -> 8 registros
  tickets_soporte      -> 8 registros

Base de datos lista para usar


In [11]:
# Cerrar conexion
conn.close()
print("Conexion cerrada. DB guardada en ../data/ecomarket.db")

Conexion cerrada. DB guardada en ../data/ecomarket.db
